# Part 3 — Churn Prediction Model
**Author:** Waqar Masood · **Student ID:** AIML_IITP_2506409

Snapshot: 2025-09-30 · Target: `churn_next_60d`

Pipeline: load → leakage check → split → baseline (LR) → strong model (GBM) → evaluation → threshold selection → interpretability → error analysis → save artifacts.

## 1. Imports & data

In [8]:
import json, joblib, numpy as np, pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score, confusion_matrix,
                              precision_recall_curve, roc_curve)
import matplotlib.pyplot as plt

DATA = Path('data')  # adjust if running from repo root
df = pd.read_csv(DATA/'rfm_modeling_snapshot.csv')
print(df.shape); df.head()

(2400, 29)


,customer_id,snapshot_date,city_tier,age_group,acquisition_channel,loyalty_tier,preferred_category,marketing_consent,recency_days,frequency_180d,...,sessions_30d,product_views_30d,cart_adds_30d,wishlist_adds_30d,abandoned_carts_30d,email_opens_30d,campaign_clicks_30d,last_visit_days_ago,churn_next_60d,split
0,CUST00001,2025-09-30,Tier 1,18-24,Instagram,Silver,Makeup,Yes,107,1,...,1,4,0,0,0,2,0,20,1,train
1,CUST00002,2025-09-30,Tier 2,25-34,Marketplace,Silver,Hair Care,Yes,40,1,...,8,31,4,2,3,0,0,0,0,train
2,CUST00003,2025-09-30,Tier 1,25-34,Influencer,NaN,Skin Care,Yes,171,1,...,1,3,0,0,0,0,0,26,1,train
3,CUST00004,2025-09-30,Tier 3,25-34,Google Search,NaN,Fragrance,No,131,1,...,1,6,0,0,0,0,0,14,1,train
4,CUST00005,2025-09-30,Tier 3,35-44,Organic,Gold,Hair Care,Yes,38,3,...,18,95,4,1,1,3,1,9,0,train


## 2. Leakage prevention
The modeling snapshot is built only from on/before-snapshot windows (recency, 30/90/180-day windows ending 2025-09-30). The target `churn_next_60d` is the 60-day *forward* window and is removed from features. The `split` column is metadata, not a feature. `intervention_history.csv` is intentionally NOT joined here — campaign sends may overlap the prediction window.

In [9]:
TARGET = 'churn_next_60d'
ID_COLS = ['customer_id','snapshot_date','split']
CAT = ['city_tier','age_group','acquisition_channel','loyalty_tier','preferred_category','marketing_consent']
NUM = [c for c in df.columns if c not in ID_COLS+CAT+[TARGET]]
print('numeric:', NUM)
print('categorical:', CAT)
assert TARGET not in NUM+CAT


numeric: ['recency_days', 'frequency_180d', 'monetary_180d', 'return_rate_180d', 'avg_discount_pct_180d', 'avg_rating_180d', 'category_diversity_180d', 'ticket_count_90d', 'negative_ticket_rate_90d', 'avg_resolution_hours_90d', 'days_since_signup', 'sessions_30d', 'product_views_30d', 'cart_adds_30d', 'wishlist_adds_30d', 'abandoned_carts_30d', 'email_opens_30d', 'campaign_clicks_30d', 'last_visit_days_ago']
categorical: ['city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier', 'preferred_category', 'marketing_consent']


## 3. Train / validation / test split (provided)

In [10]:
train = df[df.split=='train']; val = df[df.split=='validation']; test = df[df.split=='test']
print(len(train), len(val), len(test), 'pos rate train=', train[TARGET].mean().round(3))
def xy(d): return d[CAT+NUM], d[TARGET].astype(int)
Xtr,ytr = xy(train); Xv,yv = xy(val); Xte,yte = xy(test)

1728 336 336 pos rate train= 0.47


## 4. Preprocessing

In [11]:
pre = ColumnTransformer([
    ('num', StandardScaler(), NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT),
])

## 5. Baseline — Logistic Regression

In [12]:
baseline = Pipeline([('pre',pre),('clf',LogisticRegression(max_iter=1000, class_weight='balanced'))])
baseline.fit(Xtr,ytr)
pv = baseline.predict_proba(Xv)[:,1]
print('val ROC-AUC', roc_auc_score(yv,pv).round(3), 'PR-AUC', average_precision_score(yv,pv).round(3))

AttributeError: 'float' object has no attribute 'round'

## 6. Strong model — Gradient Boosting

In [ ]:
strong = Pipeline([('pre',pre),('clf',GradientBoostingClassifier(random_state=42, n_estimators=300, max_depth=3, learning_rate=0.05))])
strong.fit(Xtr,ytr)
pv_s = strong.predict_proba(Xv)[:,1]
print('val ROC-AUC', roc_auc_score(yv,pv_s).round(3), 'PR-AUC', average_precision_score(yv,pv_s).round(3))

## 7. Threshold selection
We pick the threshold that maximises F1 on the validation set, biased toward recall because missing a churner (lost LTV ≈ ₹2–5k) costs ~10–30× more than an unnecessary retention offer (~₹50).

In [ ]:
prec, rec, thr = precision_recall_curve(yv, pv_s)
f1s = 2*prec*rec/(prec+rec+1e-9)
i = int(np.argmax(f1s[:-1]))
chosen_thr = float(thr[i])
if rec[i] < 0.6:
    cand = np.where(rec[:-1] >= 0.7)[0]
    if len(cand): chosen_thr = float(thr[cand[-1]])
print('chosen threshold =', round(chosen_thr,3))
plt.figure(); plt.plot(rec, prec); plt.xlabel('recall'); plt.ylabel('precision'); plt.title('Val PR curve'); plt.show()

## 8. Evaluation on test set

In [ ]:
def report(model, X, y, t):
    p = model.predict_proba(X)[:,1]; yhat = (p>=t).astype(int)
    tn,fp,fn,tp = confusion_matrix(y, yhat).ravel()
    return dict(acc=accuracy_score(y,yhat), prec=precision_score(y,yhat,zero_division=0),
                rec=recall_score(y,yhat,zero_division=0), f1=f1_score(y,yhat,zero_division=0),
                roc=roc_auc_score(y,p), pr=average_precision_score(y,p),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))
print('Baseline @0.5 :', report(baseline, Xte, yte, 0.5))
print('Strong   @thr:', report(strong,   Xte, yte, chosen_thr))

## 9. Feature importance

In [ ]:
ohe = strong.named_steps['pre'].named_transformers_['cat']
feat = NUM + ohe.get_feature_names_out(CAT).tolist()
imp = strong.named_steps['clf'].feature_importances_
top = pd.DataFrame({'feature':feat,'importance':imp}).sort_values('importance',ascending=False).head(15)
print(top)
top.iloc[::-1].plot.barh(x='feature', y='importance', figsize=(7,6), legend=False); plt.title('Top 15 GBM features'); plt.tight_layout(); plt.show()

## 10. Error analysis preview
Full narrative in `error_analysis.md`.

In [ ]:
pte = strong.predict_proba(Xte)[:,1]
yhat = (pte>=chosen_thr).astype(int)
t2 = test.copy(); t2['proba']=pte; t2['pred']=yhat
print('FP examples'); print(t2[(t2.pred==1)&(t2[TARGET]==0)].sort_values('proba',ascending=False).head()[['customer_id','proba','recency_days','frequency_180d','sessions_30d']])
print('FN examples'); print(t2[(t2.pred==0)&(t2[TARGET]==1)].sort_values('proba').head()[['customer_id','proba','recency_days','frequency_180d','sessions_30d']])

## 11. Save artifacts

In [ ]:
joblib.dump({'model':strong,'threshold':chosen_thr,'features':{'num':NUM,'cat':CAT}}, 'model.pkl')
print('saved model.pkl')